# C12-classical-models — Practice p18 — Solution


Both pipelines fit separate scalers on the same training matrix. Logistic output is audited as probability; the SVM decision score is audited through signed margins and hinge loss.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X_train_p18 = np.array([[-3.,0.],[-2.,1.],[-1.,-1.],[-0.5,1.5],
                        [0.5,-1.],[1.,1.],[2.,-0.5],[3.,1.]], dtype=np.float64)
y_train_p18 = np.array([0,0,0,0,1,1,1,1], dtype=np.int64)
X_validation_p18 = np.array([[-1.5,0.2],[0.,0.],[1.5,0.4],[2.5,-1.]], dtype=np.float64)
y_validation_p18 = np.array([0,0,1,1], dtype=np.int64)


def fit_linear_comparison(X_train, y_train, X_validation):
    logistic = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=20260804))
    svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0))
    logistic.fit(X_train, y_train)
    svm.fit(X_train, y_train)
    lp = logistic.predict_proba(X_validation)[:, 1]
    ls = logistic.decision_function(X_validation)
    ss = svm.decision_function(X_validation)
    return {"logistic_pipeline": logistic, "svm_pipeline": svm,
            "logistic_probabilities": lp, "logistic_scores": ls,
            "logistic_predictions": (lp >= 0.5).astype(np.int64),
            "svm_scores": ss, "svm_predictions": (ss >= 0.0).astype(np.int64)}


comparison_p18 = fit_linear_comparison(X_train_p18, y_train_p18, X_validation_p18)
t_validation_p18 = 2*y_validation_p18 - 1
logistic_accuracy_p18 = float(np.mean(comparison_p18["logistic_predictions"] == y_validation_p18))
svm_accuracy_p18 = float(np.mean(comparison_p18["svm_predictions"] == y_validation_p18))
logistic_brier_p18 = float(np.mean((comparison_p18["logistic_probabilities"] - y_validation_p18)**2))
svm_margins_p18 = t_validation_p18 * comparison_p18["svm_scores"]
svm_mean_hinge_p18 = float(np.maximum(0.0, 1.0-svm_margins_p18).mean())
comparison_axes_p18 = '''Logistic regression optimizes log loss and supplies modeled probabilities; the linear SVM optimizes a margin objective and supplies signed scores. Both have linear boundaries in scaled feature space and both require train-only scaling. Here their validation labels agree, but that does not make SVM scores probabilities or establish calibration. A probability metric such as Brier complements accuracy for logistic output, while signed margins and hinge loss audit the SVM.'''


### Answer check


In [ ]:
ATOL = 1e-10
RTOL = 1e-8
assert np.allclose(comparison_p18["logistic_probabilities"], [0.25800279480818605,0.5122539768388392,0.7331579579484134,0.8860538454207187], atol=ATOL, rtol=RTOL)
assert np.allclose(comparison_p18["svm_scores"], [-1.2244429557207193,0.11681299441806298,1.177717757953494,2.663740896430869], atol=ATOL, rtol=RTOL)
assert np.array_equal(comparison_p18["logistic_predictions"], [0,1,1,1])
assert np.array_equal(comparison_p18["svm_predictions"], [0,1,1,1])
assert set(comparison_p18) == {"logistic_pipeline","svm_pipeline","logistic_probabilities","logistic_scores","logistic_predictions","svm_scores","svm_predictions"}
assert np.allclose(comparison_p18["logistic_pipeline"][0].mean_, X_train_p18.mean(axis=0), atol=ATOL, rtol=RTOL)
assert np.allclose(comparison_p18["svm_pipeline"][0].mean_, X_train_p18.mean(axis=0), atol=ATOL, rtol=RTOL)
assert np.isclose(logistic_accuracy_p18, 0.75, atol=ATOL, rtol=RTOL)
assert np.isclose(svm_accuracy_p18, 0.75, atol=ATOL, rtol=RTOL)
assert type(logistic_brier_p18) is float and type(svm_mean_hinge_p18) is float
assert isinstance(comparison_axes_p18, str) and "not" in comparison_axes_p18
